# Topic 0.1 — NumPy vectors & matrices (LESSON)

Read each section, **run every cell**, and match the output to the code.
Then go earn your checkmarks in `drills.ipynb`.

**Why this topic exists:** in ML, *everything* is a vector or a matrix —
- one data sample → a vector of features, shape `(n_features,)`
- a whole dataset → a matrix, one row per sample, shape `(n_samples, n_features)`
- model weights → a vector (or matrix)

> Journey rule #7: every library method used gets a comment explaining what it does.
> Journey rule #8: every drill maps to a lesson section — the drill map is at the bottom.

In [1]:
import time          # stdlib module for measuring time — we'll use it to race Python vs NumPy
import numpy as np   # NumPy: the array library ALL of ML is built on; 'np' is the universal alias

# np.random.default_rng() -> creates a random number Generator object (the modern NumPy way).
# seed=42 fixes the randomness: same seed = same "random" numbers every run -> reproducible results.
rng = np.random.default_rng(seed=42)

## 1. A vector is just an array with a shape

In [2]:
# np.array(list) -> converts a plain Python list into an ndarray, NumPy's core data structure.
# Unlike a list, all elements share one dtype (here float64) and live in one contiguous block
# of memory — that layout is what makes NumPy fast.
v = np.array([2.0, -1.0, 3.0])
M = np.array([[1.0, 2.0, 3.0],      # a list of lists becomes a 2-D array (matrix)
              [4.0, 5.0, 6.0]])

# .shape -> a tuple with the size of each dimension. THE attribute you'll inspect most in ML —
# 90% of NumPy/PyTorch bugs are shape bugs.
print(f"v       = {v},  shape={v.shape}")   # (3,)   1-D: a vector
print(f"M shape = {M.shape}")               # (2, 3) 2-D: a matrix, 2 rows x 3 cols

v       = [ 2. -1.  3.],  shape=(3,)
M shape = (2, 3)


## 2. The dot product — the single most-used operation in ML

Multiply elementwise, then sum. That's it.

Why care? A linear model's prediction **is** a dot product: `prediction = weights · features + bias`

In [3]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

# the manual way — a plain Python loop, so we SEE what a dot product is:
manual = 0.0
for i in range(len(a)):      # len(array) -> length of the first dimension (here: 3 elements)
    manual += a[i] * b[i]    # multiply element-by-element, accumulate the sum

print(f"by loop : {manual}")            # 1*4 + 2*5 + 3*6 = 32

# np.dot(a, b) -> the same multiply-and-sum, but computed in optimized C instead of Python
print(f"np.dot  : {np.dot(a, b)}")

# a @ b -> the @ operator is Python's matrix-multiplication operator; for two 1-D arrays
# it IS the dot product. Same result, nicer syntax — this is what you'll see in real code.
print(f"a @ b   : {a @ b}")

by loop : 32.0
np.dot  : 32.0
a @ b   : 32.0


## 3. Why we vectorize: loops are slow, NumPy is fast

This is THE reason ML code looks the way it does: no loops over samples, ever.
One matrix expression handles the whole dataset at once.

In [4]:
# rng.random(n) -> array of n random floats uniformly drawn from [0, 1)
big_a = rng.random(1_000_000)   # 1_000_000 is just 1000000 — underscores are readable separators
big_b = rng.random(1_000_000)

# time.perf_counter() -> high-precision clock reading (in seconds), made for benchmarking.
# Pattern: read it before and after the work; the difference is the elapsed time.
t0 = time.perf_counter()
s = 0.0
for i in range(len(big_a)):     # 1 million Python-level iterations...
    s += big_a[i] * big_b[i]    # ...each paying interpreter overhead per element
loop_ms = (time.perf_counter() - t0) * 1000   # seconds -> milliseconds

t0 = time.perf_counter()
s2 = big_a @ big_b              # the SAME 1M multiply-adds, but in one C call (vectorized)
numpy_ms = (time.perf_counter() - t0) * 1000

print(f"python loop: {loop_ms:8.1f} ms")
print(f"numpy @    : {numpy_ms:8.2f} ms   -> ~{loop_ms / numpy_ms:.0f}x faster")

python loop:    390.7 ms
numpy @    :     2.77 ms   -> ~141x faster


## 4. Matrix multiplication = many dot products at once — and the linear model

**The shape rule:** `(a, b) @ (b, c) -> (a, c)`. Inner dims must MATCH.

In [5]:
# matrix @ vector: (2,3) @ (3,) -> (2,) — each ROW of M gets dotted with v
print(f"M @ v = {M @ v}")

# M[0] -> row indexing: the first row of M as a 1-D array (like list indexing, but per-row)
print(f"  row0 . v = {M[0] @ v},  row1 . v = {M[1] @ v}  (same numbers!)")

# rng.random((rows, cols)) -> pass a shape TUPLE to get a 2-D array of random floats
A = rng.random((4, 3))
B = rng.random((3, 5))
print(f"(4,3) @ (3,5) -> {(A @ B).shape}")   # inner 3s match and disappear -> (4, 5)

M @ v = [ 9. 21.]
  row0 . v = 9.0,  row1 . v = 21.0  (same numbers!)
(4,3) @ (3,5) -> (4, 5)


**Now the payoff — a whole linear model in one line.** A linear model predicts
`w·x + b` for ONE sample. With matrix @ vector, we predict for EVERY sample at once:

In [6]:
X = np.array([[1.0, 2.0],           # a tiny "dataset": 3 samples (rows), 2 features (cols)
              [3.0, 4.0],
              [5.0, 6.0]])
w = np.array([0.5, -1.0])           # one weight per feature -> shape (2,)
bias = 2.0                          # one scalar bias

# X @ w -> (3,2) @ (2,) = (3,): each row of X dotted with w = one prediction per sample.
# + bias -> the scalar broadcasts to all 3 predictions (see section 5 for broadcasting).
predictions = X @ w + bias
print(f"predictions = {predictions}, shape={predictions.shape}")
# sample 0: 0.5*1 + (-1)*2 + 2 = 0.5 — check it by hand! This ONE line is what
# every linear model, and every layer of a neural network, computes at its core.

predictions = [ 0.5 -0.5 -1.5], shape=(3,)


## 5. Broadcasting — NumPy stretches shapes for you

**The rule:** compare shapes right-to-left; dims are compatible if **equal or 1**.

```
(3, 2) vs (2,)  -> (2,) acts like (1, 2) -> stretched to (3, 2). OK!
(3, 2) vs (3,)  -> ERROR: 2 vs 3 mismatch on the last axis.
```

**Gotcha:** to broadcast per-ROW you need a *column* shape `(3, 1)` — that's what `keepdims=True` is for.

In [7]:
# (X from section 4: 3 samples, 2 features)
# array * scalar -> the scalar is broadcast (stretched) to every element; no loop needed
print(f"X * 10 :\n{X * 10}")

# X.mean(axis=0) -> mean along axis 0. Axis 0 runs DOWN the rows, so axis=0 collapses the
# rows and leaves one mean PER COLUMN -> shape (2,). Rule of thumb: the axis you pass in
# is the axis that DISAPPEARS.
col_means = X.mean(axis=0)
print(f"col_means = {col_means}")

# (3,2) - (2,): the (2,) row is broadcast across all 3 rows -> centers the dataset in one line
print(f"X - col_means :\n{X - col_means}")

# X.sum(axis=1) alone would give shape (3,) — and (3,2) / (3,) ERRORS (right-to-left mismatch).
# keepdims=True keeps the collapsed axis as size 1 -> shape (3, 1), a COLUMN,
# which broadcasts across each row exactly like we want.
row_sums = X.sum(axis=1, keepdims=True)
print(f"X / row_sums (normalize each row):\n{X / row_sums}")

X * 10 :
[[10. 20.]
 [30. 40.]
 [50. 60.]]
col_means = [3. 4.]
X - col_means :
[[-2. -2.]
 [ 0.  0.]
 [ 2.  2.]]
X / row_sums (normalize each row):
[[0.33333333 0.66666667]
 [0.42857143 0.57142857]
 [0.45454545 0.54545455]]


## 6. Spread: `.std()` and the z-score

The mean tells you a feature's CENTER; the **standard deviation** tells you its SPREAD.
ML models care because features on wildly different scales (age 0-100 vs salary 0-1,000,000)
confuse training — so we **standardize**: shift to mean 0, scale to std 1. The result is a
**z-score**: "how many standard deviations is this value from the mean?"

In [8]:
heights = np.array([160.0, 170.0, 180.0, 190.0])   # one feature, in cm

# .mean() / .std() with NO axis argument -> computed over ALL elements, returns one scalar.
# .std() = standard deviation: sqrt of the average squared distance from the mean.
mu = heights.mean()
sigma = heights.std()
print(f"mean = {mu}, std = {sigma:.3f}")

# the z-score: subtract the center, divide by the spread. Broadcasting does it for the
# whole array at once — result has mean 0 and std 1, whatever units you started in.
z = (heights - mu) / sigma
print(f"z-scores = {z}")
print(f"check: mean(z) = {z.mean():.1f}, std(z) = {z.std():.1f}")

# On a DATASET you do the same per-column: .mean(axis=0) and .std(axis=0) (section 5's axis
# rule) give one center and one spread PER FEATURE — that's your drill 6.
print(f"per-column std of X = {X.std(axis=0)}")

# np.allclose(p, q) -> True if p and q are equal element-wise within a tiny tolerance.
# Floats accumulate rounding errors, so NEVER compare them with == ; allclose is how
# the drill checks (and real ML tests) compare your arrays against the expected ones.
print(f"0.1 + 0.2 == 0.3 ? {0.1 + 0.2 == 0.3}   (floating point!)")
print(f"np.allclose(0.1 + 0.2, 0.3) ? {np.allclose(0.1 + 0.2, 0.3)}")

mean = 175.0, std = 11.180
z-scores = [-1.34164079 -0.4472136   0.4472136   1.34164079]
check: mean(z) = 0.0, std(z) = 1.0
per-column std of X = [1.63299316 1.63299316]
0.1 + 0.2 == 0.3 ? False   (floating point!)
np.allclose(0.1 + 0.2, 0.3) ? True


## 7. Making new axes: `a[:, None]` — broadcasting in 2-D

Broadcasting gets REALLY powerful when you combine a column with a row: NumPy stretches
BOTH and you get a full table of every-pair combinations — with zero loops.
This pattern later computes distance matrices for KNN and k-means.

In [9]:
nums = np.array([1.0, 2.0, 3.0])    # shape (3,)

# a[:, None] -> adds a NEW axis of size 1. The ':' keeps the existing elements,
# 'None' (alias np.newaxis) inserts the axis -> shape goes (3,) -> (3, 1): a COLUMN.
col = nums[:, None]
print(f"nums shape {nums.shape} -> col shape {col.shape}")

# a[None, :] -> same trick on the other side -> shape (1, 3): a ROW.
row = nums[None, :]
print(f"row shape = {row.shape}")

# column (3,1) * row (1,3): broadcasting stretches the 1-sized dims of BOTH
# -> a (3,3) table where cell [i,j] = nums[i] * nums[j]. A multiplication table, no loops!
table = col * row
print(f"multiplication table:\n{table}")
# Swap * for - or + and you get every-pair differences/sums — that's drill 7's shape.

nums shape (3,) -> col shape (3, 1)
row shape = (1, 3)
multiplication table:
[[1. 2. 3.]
 [2. 4. 6.]
 [3. 6. 9.]]


---
**Lesson done.** Every drill maps to a section (rule #8):

| Drill | Uses sections |
|---|---|
| 1 dot_by_hand | 2 (the loop version) |
| 2 predict | 4 (`X @ w + bias`) |
| 3 valid_shape | 4 (the shape rule) |
| 4 center_columns | 5 (`X - X.mean(axis=0)`) |
| 5 normalize_rows | 5 (`keepdims=True`) |
| 6 standardize | 6 (z-score) + 5 (axis=0) |
| 7 pairwise_diff | 7 (`a[:, None]`) |

Now open `drills.ipynb` and make every check pass. ✅